# Portfolio Project 2, Part 1: An Intent Classifier for Your Own Chatbot
### AITCD001 Chatbots Development · Learning Outcome 3 · United African Institute of Technology

**Due Monday 9 November 2026, 23:59.**

Cells marked **YOUR DOMAIN** are replaced with your own chatbot's data. Cells marked **PROVIDED**
train, evaluate and save: read them, do not change them.

The worked example is a **fictional** bus ticket desk (the Weeks 7 and 8 example). It may not be
submitted.

**Steps:** File, then Save a copy in Drive. Upload your `intents.csv`. Replace Steps 1, 2 and 6
and the `TRY` messages of Step 7 with your own, then run everything from the top (Runtime, then
Run all).

---

---

## Setup

**PROVIDED.** Run once. The same library and settings as the Week 9 notebook.

In [ ]:
# AITCD001, Portfolio Project 2, Part 1: an intent classifier for my own chatbot
import csv
import os
import re

import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import make_pipeline

import sklearn
print("ready: scikit-learn", sklearn.__version__)

ready: scikit-learn 1.6.1


---

## The worked example

**PROVIDED.** A fictional bus desk, 15 messages per label, already split. It is used only when `intents.csv` is missing, and the first line of Step 1 says so. It may not be submitted.

In [ ]:
# PROVIDED: the worked example, a FICTIONAL bus desk. It may not be submitted.
# It is written to example_bus_intents.csv, and used only when intents.csv is missing.
EXAMPLE = [
    ("ndashaka kujya musanze", "book", "train"),
    ("can i get a seat tonight", "book", "train"),
    ("get me on the morning bus", "book", "train"),
    ("book me to rubavu", "book", "train"),
    ("two tickets to huye today", "book", "train"),
    ("i want to travel to musanze tomorrow", "book", "train"),
    ("book a seat to huye", "book", "train"),
    ("booking for 3 people", "book", "train"),
    ("reserve 2 seats for rubavu", "book", "train"),
    ("one ticket to nyagatare please", "book", "train"),
    ("can i book for friday", "book", "train"),
    ("ndashaka itike ya huye", "book", "train"),
    ("cancel the trip", "cancel", "train"),
    ("i cant travel anymore", "cancel", "train"),
    ("i will not travel", "cancel", "train"),
    ("cancel my booking", "cancel", "train"),
    ("hagarika itike yanjye", "cancel", "train"),
    ("ndashaka guhagarika itike", "cancel", "train"),
    ("drop my booking", "cancel", "train"),
    ("i changed my plans, cancel", "cancel", "train"),
    ("cancel my seat to huye", "cancel", "train"),
    ("stop my reservation", "cancel", "train"),
    ("cancl my ticket", "cancel", "train"),
    ("refund please", "cancel", "train"),
    ("how much to musanze", "fares", "train"),
    ("cost of a ticket to rubavu", "fares", "train"),
    ("itike ni angahe", "fares", "train"),
    ("hw mch 2 nyagatare", "fares", "train"),
    ("do kids pay full price", "fares", "train"),
    ("is the fare cheaper at night", "fares", "train"),
    ("is there a student discount", "fares", "train"),
    ("ticket price for rubavu?", "fares", "train"),
    ("fares list", "fares", "train"),
    ("how much is a return ticket", "fares", "train"),
    ("price to huye pls", "fares", "train"),
    ("fare for a child", "fares", "train"),
    ("when is the next bus", "hours", "train"),
    ("when does the office open", "hours", "train"),
    ("what time does the first bus leave", "hours", "train"),
    ("are you open on sunday", "hours", "train"),
    ("saa ngahe bus igenda?", "hours", "train"),
    ("departure times please", "hours", "train"),
    ("what time do you close", "hours", "train"),
    ("last bus to musanze tonight?", "hours", "train"),
    ("bus to huye leaves when", "hours", "train"),
    ("r u open today", "hours", "train"),
    ("wen is the last bus", "hours", "train"),
    ("timetable pls", "hours", "train"),
    ("is there space for a suitcase", "luggage", "train"),
    ("how heavy can my suitcase be", "luggage", "train"),
    ("can i carry a bicycle", "luggage", "train"),
    ("luggage limit?", "luggage", "train"),
    ("i have a big box, is that ok", "luggage", "train"),
    ("imizigo yemewe ni ingahe?", "luggage", "train"),
    ("how many bags allowed", "luggage", "train"),
    ("big luggage ok?", "luggage", "train"),
    ("can i take a sack of potatoes", "luggage", "train"),
    ("do you charge for bags", "luggage", "train"),
    ("nfite imizigo myinshi", "luggage", "train"),
    ("is the bag fee extra", "luggage", "train"),
    ("who won the match", "out_of_scope", "train"),
    ("where is the bus park", "out_of_scope", "train"),
    ("are you hiring drivers", "out_of_scope", "train"),
    ("the driver was rude", "out_of_scope", "train"),
    ("can i pay with momo", "out_of_scope", "train"),
    ("i left my bag on the bus", "out_of_scope", "train"),
    ("the bus was dirty", "out_of_scope", "train"),
    ("ok", "out_of_scope", "train"),
    ("what is your name", "out_of_scope", "train"),
    ("asdfgh", "out_of_scope", "train"),
    ("my seat was broken", "out_of_scope", "train"),
    ("i want to speak to the manager", "out_of_scope", "train"),
    ("i want to go to rubavu on saturday", "book", "test"),
    ("reserve a ticket for sunday", "book", "test"),
    ("i need a seat on the 8am bus", "book", "test"),
    ("cancle the ticket", "cancel", "test"),
    ("i want my money back", "cancel", "test"),
    ("call off my seat", "cancel", "test"),
    ("how many francs to musanze", "fares", "test"),
    ("what does a ticket cost", "fares", "test"),
    ("itike ya huye ni angahe", "fares", "test"),
    ("bus igenda saa ngahe ejo", "hours", "test"),
    ("is there a bus at 6am", "hours", "test"),
    ("how early do buses start", "hours", "test"),
    ("can i bring my guitar", "luggage", "test"),
    ("can i bring two bags", "luggage", "test"),
    ("can i send a parcel on the bus", "luggage", "test"),
    ("is there wifi on the bus", "out_of_scope", "test"),
    ("i lost my phone on the bus", "out_of_scope", "test"),
    ("tell me a joke", "out_of_scope", "test"),
]

with open("example_bus_intents.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["text", "intent", "source", "split"])
    for text, intent, split in EXAMPLE:
        writer.writerow([text, intent, "example", split])
print(len(EXAMPLE), "example rows written")

90 example rows written


---

## Step 1: Your dataset

**YOUR DOMAIN.** Improve the dataset first, from the written feedback on Assignment 8: in `Week_8_Take_Home_Intent_Dataset.ipynb`, edit Step 3, run every cell again, and download the new `intents.csv`. Then upload it here, to the Files panel on the left.

In [ ]:
# STEP 1. YOUR DOMAIN: the chatbot's name. Upload intents.csv (Assignment 8, improved after
# feedback) to the Files panel first.
BOT_NAME = "ULK Exam & Registry Desk"

FILE = "intents.csv"
if not os.path.exists(FILE):
    print("intents.csv NOT FOUND: the worked example is used. It may not be submitted.")
    FILE = "example_bus_intents.csv"

df = pd.read_csv(FILE)
train = df[df["split"] == "train"]
test = df[df["split"] == "test"]
train_texts = list(train["text"])
train_labels = list(train["intent"])
test_texts = list(test["text"])
test_labels = list(test["intent"])

print(BOT_NAME, "| file:", FILE)
print(len(train_texts), "training,", len(test_texts), "test,",
      df["intent"].nunique(), "labels:", sorted(set(train_labels)))

intents.csv NOT FOUND: the worked example is used. It may not be submitted.
ULK Exam & Registry Desk | file: example_bus_intents.csv
72 training, 18 test, 6 labels: ['book', 'cancel', 'fares', 'hours', 'luggage', 'out_of_scope']


---

## Step 2: The rules, as the baseline

**YOUR DOMAIN.** The classifier must beat the rules on the same test set, so they are scored here again: the split may have changed since Assignment 8.

In [ ]:
# STEP 6. YOUR DOMAIN: paste the RULES list from Step 2 of your Portfolio Project 1.
RULES = [
    ("cancel",     r"\bcancel\w*\b|\bcall it off\b|\bstop\b"),
    ("book",       r"\bbook\w*\b|\bappointments?\b|\breserv\w*\b|\bregistry\b|\bslot\b"),
    ("timetable",  r"\btimetables?\b|\bexam\w*\b|\bschedules?\b|\bwhat ?time\b|\bibizamini\b"),
    ("fees",       r"\bfees?\b|\bclearance\b|\bprices?\b|\bcosts?\b|\bhow ?much\b|\bangahe\b"),
    ("transcript", r"\btranscripts?\b|\bresults?\b|\bgrades?\b|\bmarks?\b|\bamanota\b"),
]

print(len(RULES), "rules")

5 rules


In [ ]:
# STEP 2, continued. PROVIDED: the baseline, the rules on the SAME test set.
def clean(text):
    """Remove punctuation (keeping the colon of 21:00), lowercase, trim."""
    return re.sub(r"[^\w\s:]", "", text).lower().strip()


def rule_label(text):
    """The first rule that matches, or out_of_scope."""
    for label, pattern in RULES:
        if re.search(pattern, clean(text)):
            return label
    return "out_of_scope"


def outcomes(guesses, labels):
    """Right; missed (a real request sent to the fallback); confident and wrong."""
    right = missed = wrong = 0
    for guess, label in zip(guesses, labels):
        if guess == label:
            right += 1
        elif guess == "out_of_scope":
            missed += 1
        else:
            wrong += 1
    return right, missed, wrong


rules_result = outcomes([rule_label(t) for t in test_texts], test_labels)
print("Rules: %d right, %d missed, %d confident and wrong, of %d" %
      (rules_result + (len(test_texts),)))

Rules: 4 right, 12 missed, 2 confident and wrong, of 18


---

## Step 3: Words and characters

**PROVIDED.** Two classifiers, the same model on different features (Week 9, Sections 13 to 19). The one with the higher test accuracy is kept.

In [ ]:
# STEP 3. PROVIDED: words and characters, trained on the training set only; the better is kept.
CANDIDATES = {
    "words": TfidfVectorizer(),
    "characters": TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4)),
}

accuracy = {}
models = {}
for name, vectorizer in CANDIDATES.items():
    model = make_pipeline(vectorizer, LogisticRegression(max_iter=1000))
    model.fit(train_texts, train_labels)
    models[name] = model
    accuracy[name] = accuracy_score(test_labels, model.predict(test_texts))
    print(f"{name:11} accuracy {accuracy[name]:.0%}")

best = max(accuracy, key=accuracy.get)
classifier = models[best]
predicted = classifier.predict(test_texts)
print("kept:", best)

words       accuracy 56%
characters  accuracy 67%
kept: characters


---

## Step 4: Every intent, and the confusion matrix

**PROVIDED.** Precision: of the times the classifier said an intent, the share that was right. Recall: of the messages of that intent, the share it found. The matrix shows which intents are confused.

In [ ]:
# STEP 4. PROVIDED: every intent, then the confusion matrix (as in the Week 9 notebook).
from sklearn.metrics import classification_report

print(classification_report(test_labels, predicted,
                            zero_division=0))

              precision    recall  f1-score   support

        book       0.67      0.67      0.67         3
      cancel       0.67      0.67      0.67         3
       fares       1.00      1.00      1.00         3
       hours       1.00      0.67      0.80         3
     luggage       1.00      0.33      0.50         3
out_of_scope       0.33      0.67      0.44         3

    accuracy                           0.67        18
   macro avg       0.78      0.67      0.68        18
weighted avg       0.78      0.67      0.68        18



In [ ]:
pd.set_option("display.width", 200)      # all nine columns,
pd.set_option("display.max_columns", 20)  # on one line
actual = pd.Series(test_labels, name="actual")
guessed = pd.Series(predicted, name="predicted")
print(pd.crosstab(actual, guessed))

predicted     book  cancel  fares  hours  luggage  out_of_scope
actual                                                         
book             2       0      0      0        0             1
cancel           0       2      0      0        0             1
fares            0       0      3      0        0             0
hours            0       0      0      2        0             1
luggage          0       1      0      0        1             1
out_of_scope     1       0      0      0        0             2


---

## Step 5: The threshold table

**PROVIDED.** Below the threshold, the chatbot falls back instead of answering. Each line shows what one threshold costs: right answers, real requests missed, confident wrong answers.

In [ ]:
# STEP 5. PROVIDED: the threshold table (as in the Week 9 notebook).
scores = classifier.predict_proba(test_texts)
classes = classifier.classes_

print("threshold  right  fallbacks  missed  wrong")
for threshold in [0.0, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4]:
    right = fallbacks = missed = wrong = 0
    for row, label in zip(scores, test_labels):
        best = row.argmax()
        intent = classes[best]
        if row[best] < threshold:          # not sure enough
            intent = "out_of_scope"
        if intent == "out_of_scope":
            fallbacks += 1
        if intent == label:
            right += 1
        elif intent == "out_of_scope":
            missed += 1
        else:
            wrong += 1
    print(f"{threshold:9} {right:6} {fallbacks:10} {missed:7} {wrong:6}")

threshold  right  fallbacks  missed  wrong
      0.0     12          6       4      2
      0.1     12          6       4      2
     0.15     12          6       4      2
      0.2     13          7       4      1
     0.25      9         11       8      1
      0.3      4         17      14      0
      0.4      3         18      15      0


---

## Step 6: Choose the threshold, and say why

**YOUR DOMAIN.** Choose from the table, for your users: what does a wrong answer cost them, and what does a fallback cost? The reason goes in `WHY` and in the README.

In [ ]:
# STEP 6. YOUR DOMAIN: the threshold, and why. Read the table above first.
THRESHOLD = 0.2
WHY = ("Most right answers with the fewest confident wrong ones; a wrong booking "
       "costs a traveller a trip, so a wrong answer is worse than a fallback.")

# PROVIDED: the classifier at that threshold, against the rules, on the same test set
final = []
for row in classifier.predict_proba(test_texts):
    best_i = row.argmax()
    intent = classifier.classes_[best_i]
    if row[best_i] < THRESHOLD:
        intent = "out_of_scope"
    final.append(intent)

model_result = outcomes(final, test_labels)
fallback_rate = final.count("out_of_scope") / len(final)
print(f"threshold {THRESHOLD}: {WHY}")
print(f"{'':12}{'right':>7}{'missed':>8}{'wrong':>7}")
print(f"{'rules':12}{rules_result[0]:7}{rules_result[1]:8}{rules_result[2]:7}")
print(f"{'classifier':12}{model_result[0]:7}{model_result[1]:8}{model_result[2]:7}")
print(f"fallback rate: {fallback_rate:.0%}")

threshold 0.2: Most right answers with the fewest confident wrong ones; a wrong booking costs a traveller a trip, so a wrong answer is worse than a fallback.
              right  missed  wrong
rules             4      12      2
classifier       13       4      1
fallback rate: 39%


---

## Step 7: Save the classifier

**PROVIDED.** `intent_classifier.joblib` holds the vectorizer and the model together. Part 2 loads it into the Telegram chatbot. Replace the four `TRY` messages with your own.

In [ ]:
# STEP 7. PROVIDED: save the classifier for Part 2 (Telegram), load it back, try it.
joblib.dump(classifier, "intent_classifier.joblib")
loaded = joblib.load("intent_classifier.joblib")

# YOUR DOMAIN: four messages your users might send, none of them in the dataset
TRY = [
    "wen r final exams starting",
    "hw mch 4 lost id replacement",
    "can i book registry 2moro morning",
    "where is the campus cafeteria",
]
for m in TRY:
    row = loaded.predict_proba([m])[0]
    best_i = row.argmax()
    intent = loaded.classes_[best_i]
    verdict = "answer" if row[best_i] >= THRESHOLD and intent != "out_of_scope" else "fallback"
    print(f"{m:34} {intent:13} {row[best_i]:.2f}  {verdict}")

wen r final exams starting         hours         0.21  answer
hw mch 4 lost id replacement       fares         0.28  answer
can i book registry 2moro morning  book          0.30  answer
where is the campus cafeteria      out_of_scope  0.26  fallback


---

## Step 8: The README, then publish and submit

Add a section **The classifier** to `dataset/README.md`:

| | |
|---|---|
| Features kept | words or characters, and both accuracies |
| Rules on the same test set | right, missed, wrong |
| Classifier at the threshold | right, missed, wrong, the fallback rate |
| Threshold | the value, and why (Step 6) |
| Weakest intent | the lowest F1, and what data would help it |
| What changed after the Assignment 8 feedback | two or three lines |

1. In the `dataset` folder of your Portfolio Project 2 repository, upload this notebook
   (File, then Download, then .ipynb), the improved `intents.csv` and `intent_classifier.joblib`.
2. Open the folder in a private browsing window. Paste its URL in Google Classroom and click
   **Turn in**, by **Monday 9 November 2026, 23:59**.